# let's explore inner nodes

In [1]:
!ls /data-ssd/alex/datasets/vision-export-20240512050009-aka-2.14/

all_taxa.csv		   taxonomy.csv		 taxon_range_recalls.csv
geo_spatial_grid_datasets  taxonomy_visual.txt	 test_data.csv
iconic_taxa.csv		   taxon_range_csvs	 train_data.csv
spatial_data.csv	   taxon_range_csvs.tgz  val_data.csv


In [2]:
from collections import defaultdict
import math
import os
from pathlib import Path

import h3pandas
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

from lib.geo_dataset_maker import DiscretizedInatGeoModelDataset

2024-07-28 10:10:44.583068: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-28 10:10:44.583095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-28 10:10:44.584036: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-28 10:10:44.588914: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-28 10:10:45.074465: W tensorflow/compiler/tf2

In [3]:
params = {
    "export_dir": "/data-ssd/alex/datasets/vision-export-20240512050009-aka-2.14",
    "export_short_version": "2.14-inner",
    "train_only_cid_data": True,
    "train_only_wild_data": False,
    "h3_resolution": 6,
    "num_random_samples": 100_000,
    "elevation_file": "/home/alex/elevation_h3_resolution6.csv",
    "batch_size": 1024,
    "shuffle_buffer_size": 50_000,
    "full_shuffle_before_tfrecords": False,
}


In [4]:
taxonomy_path = Path("/") / params["export_dir"] / "taxonomy.csv"
tax = pd.read_csv(taxonomy_path)
leaf_tax = tax[~tax.leaf_class_id.isna()]
num_leaf_taxa = len(leaf_tax)

In [5]:
tax[tax.taxon_id==87283]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name
15814,1541885.0,87283,20.0,NaN,NaN,94108,Centrolabrus


In [6]:
tax[tax.parent_taxon_id==87283]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name
15815,87283.0,96913,10.0,913.0,5.0,913,Centrolabrus exoletus


### double check that each taxon only occurs once as a spatial class id

In [7]:
for i in tqdm(range(len(tax))):
    t = tax[tax.spatial_class_id == i]
    if len(t) != 1:
        print(i)

  0%|          | 0/133675 [00:00<?, ?it/s]

In [8]:
tax[tax.spatial_class_id==93154]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name
78697,153097.0,713245,25.0,NaN,NaN,93154,Ascalaphini


In [9]:
tax[tax.parent_taxon_id==87283]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name
15815,87283.0,96913,10.0,913.0,5.0,913,Centrolabrus exoletus


In [10]:
range(len(tax))

range(0, 133675)

In [11]:
tax[tax.taxon_id==595706]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name
84435,51876.0,595706,10.0,2401.0,2.0,2401,Trifolium diffusum


# load the spatial data

In [12]:
spatial_data_path = Path("/") / params["export_dir"] / "spatial_data.csv"
spatial_train = pd.read_csv(
    spatial_data_path,
    usecols=[
        "latitude",
        "longitude",
        "taxon_id",
        "community",
        "captive",
        "spatial_class_id",
    ],
    dtype={
        "latitude": float,
        "longitude": float,
        "taxon_id": int,
        "community": bool,
        "captive": bool,
        "spatial_class_id": int,
    },
)

In [13]:
# no data at life
assert(len(spatial_train[spatial_train.taxon_id == 48460]) == 0)

# dataset cleaning

In [14]:
# skip this step, we want data at these inner nodes
# spatial_train = spatial_train[
#     spatial_train.taxon_id.isin(spatial_train.leaf_tax.taxon_id.unique())
# ]

In [15]:
spatial_train = spatial_train[
    spatial_train.community == True
]

In [16]:
spatial_train = spatial_train.rename(
    {"latitude": "lat", "longitude": "lng"}, axis=1
)

In [17]:
spatial_train = spatial_train.rename(
    {"true_spatial_class_id": "spatial_class_id"}, axis=1
)

In [18]:
num_obs = spatial_train.shape[0]
spatial_train = spatial_train[
    (
        (spatial_train["lat"] <= 90)
        & (spatial_train["lat"] >= -90)
        & (spatial_train["lng"] <= 180)
        & (spatial_train["lng"] >= -180)
    )
]
if (num_obs - spatial_train.shape[0]) > 0:
    print(
        " ",
        num_obs - spatial_train.shape[0],
        "items filtered due to invalid locations",
    )


In [19]:
# drop null island locations
num_obs = spatial_train.shape[0]
spatial_train = spatial_train[
    ((spatial_train["lat"] != 0) & (spatial_train["lng"] != 0))
]
if (num_obs - spatial_train.shape[0]) > 0:
    print(
        " ",
        num_obs - spatial_train.shape[0],
        "observation(s) with a 0 lat and lng entry out of",
        num_obs,
        "removed",
    )

  646 observation(s) with a 0 lat and lng entry out of 137510793 removed


In [20]:
# drop observations identified to 48460
# that's life - it's not meaningful to train on this

num_obs = spatial_train.shape[0]
spatial_train = spatial_train[spatial_train.taxon_id != 48460]
if (num_obs - spatial_train.shape[0]) > 0:
    print(
        " ",
        num_obs - spatial_train.shape[0],
        "observation(s) identified as \'Life\' out of",
        num_obs,
        "removed",
    )

In [21]:
spatial_train

,lat,lng,spatial_class_id,taxon_id,community,captive
0,28.0000,113.0000,0,145559,True,False
1,30.2540,120.1235,0,145559,True,False
2,35.5684,139.5006,0,145559,True,False
3,35.7377,139.5967,0,145559,True,False
4,35.6196,139.3929,0,145559,True,False
...,...,...,...,...,...,...
190904750,10.5193,-84.6882,133674,47734,True,False
190904751,2.1928,-75.6825,133674,47734,True,False
190904752,8.9862,-79.5469,133674,47734,True,False
190904753,30.7811,-86.5763,133674,47734,True,False


In [22]:
spatial_train = spatial_train[["lat", "lng", "spatial_class_id", "taxon_id"]]

In [23]:
assert spatial_train.isna().any().any() == False

In [24]:
spatial_train.sample(10)

,lat,lng,spatial_class_id,taxon_id
21070425,-24.2418,-46.9943,12647,100398
129456722,27.9476,-111.0769,77096,164413
14907864,54.1024,45.1035,9176,129115
107838794,45.7182,-60.6080,63716,126658
64515,47.3721,6.8591,70,133517
71884174,9.6521,-84.6605,43079,145238
93833460,-37.7374,145.0569,55862,339919
185005426,36.1132,-84.4827,128696,787564
104848088,48.4063,-123.3228,61918,154762
84895574,-7.9238,-35.1650,50585,807514


# convert to discretized h3

In [25]:
print("  adding h3 index")
h3_resolution = params.get("h3_resolution")
assert h3_resolution != None, "Must define h3 resolution for training"

h3_index_name = "h3_0{}".format(h3_resolution)
spatial_train_h3 = spatial_train.h3.geo_to_h3(h3_resolution)

print("  doing h3 pivot")
spatial_train_h3_dense = spatial_train_h3.pivot_table(
    index=h3_index_name, values="spatial_class_id", aggfunc=set
)

spatial_train_h3_dense[
    "spatial_class_id"
] = spatial_train_h3_dense.spatial_class_id.apply(lambda x: list(x))


  adding h3 index
  doing h3 pivot


# make our taxonomy

In [26]:
class Taxon:
    
    def is_leaf(self):
        return false

    def __init__(self, tax_row):
        self.name = tax_row["name"]
        self.taxon_id = int(tax_row.taxon_id)

        # convert from pandas nans to python None
        if tax_row.parent_taxon_id is None or np.isnan(tax_row.parent_taxon_id):
            self.parent_taxon_id = None
        else:
            self.parent_taxon_id = int(tax_row.parent_taxon_id)

        if tax_row.spatial_class_id is None or tax_row.spatial_class_id == "" or np.isnan(tax_row.spatial_class_id):
            self.spatial_class_id = None
        else:
            self.spatial_class_id = int(tax_row.spatial_class_id)

        if tax_row.leaf_class_id is None or np.isnan(tax_row.leaf_class_id):
            self.leaf_class_id = None
        else:
            self.leaf_class_id = int(tax_row.leaf_class_id)


    def __str__(self):
        return "taxon id {} ({}) - parent taxon id {}".format(
            self.taxon_id,
            self.name,
            self.parent_taxon_id,
        )

    def is_leaf(self):
        if self.leaf_class_id is None:
            return False
        else:
            return True

class Taxonomy:
    
    def __init__(self, tax_df):
        self.all_taxa = []
        self.taxon_id_to_taxon = {}
        self.taxon_id_to_children = defaultdict(list)
        self.spatial_id_to_taxon = {}
        
        # convert from pandas to python objects
        for _, taxon_row in tax_df.iterrows():
            taxon = Taxon(taxon_row)
            self.all_taxa.append(taxon)

        # setup lookup dictionaries
        for taxon in self.all_taxa:
            self.taxon_id_to_taxon[taxon.taxon_id] = taxon
            self.taxon_id_to_children[taxon.parent_taxon_id].append(taxon)
            self.spatial_id_to_taxon[taxon.spatial_class_id] = taxon

tax_df = pd.read_csv(
    Path("/") / params["export_dir"] / "taxonomy.csv",
)
taxonomy = Taxonomy(tax_df)
print(taxonomy.all_taxa[15])

taxon id 199327 (Ascidiella aspersa) - parent taxon id 171223


# inject inner nodes

In [27]:
str(taxonomy.spatial_id_to_taxon[93154])

'taxon id 713245 (Ascalaphini) - parent taxon id 153097'

In [28]:
tax[tax.taxon_id==87283]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name
15814,1541885.0,87283,20.0,NaN,NaN,94108,Centrolabrus


In [29]:
tax[tax.parent_taxon_id=="87283"]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name


In [30]:
len(spatial_train)

137510147

In [31]:
spatial_train[spatial_train.taxon_id=="87283"]

,lat,lng,spatial_class_id,taxon_id


In [32]:
spatial_train[spatial_train.spatial_class_id==96913].sample(3)

,lat,lng,spatial_class_id,taxon_id
148301162,33.7710,-118.1353,96913,1540556
148301164,34.4075,-119.8510,96913,1540556
148301163,37.4789,-122.4414,96913,1540556


In [33]:
# iterate over the spatial grid cells
for i in tqdm(range(len(spatial_train_h3_dense))):
    row = spatial_train_h3_dense.iloc[i]  

    # cids that were directly reported in a grid cell
    class_ids = row.spatial_class_id
    
    ascendents_class_ids = []
    # find the ascendents of each of the nodes reported in the cell
    for cid in class_ids:
        taxon = taxonomy.spatial_id_to_taxon[cid]

        if taxon.parent_taxon_id is None:
            continue

        parent = taxonomy.taxon_id_to_taxon[taxon.parent_taxon_id]
        ascendents_class_ids.append(parent.spatial_class_id)
        while parent.parent_taxon_id is not None:
            parent = taxonomy.taxon_id_to_taxon[parent.parent_taxon_id]
            if parent.parent_taxon_id is not None:
                ascendents_class_ids.append(parent.spatial_class_id)
    
    # combine the original class ids with the found ascendents    
    new_set = list(set(ascendents_class_ids + class_ids))
    spatial_train_h3_dense.loc[row.name, "spatial_class_id"] = new_set

  0%|          | 0/849535 [00:00<?, ?it/s]

In [ ]:
# for i in tqdm(range(len(spatial_train_h3_dense))):
#     row = spatial_train_h3_dense.iloc[i]  
#     class_ids = row.spatial_class_id
#     for cid in class_ids:
#         taxon = taxonomy.spatial_id_to_taxon[cid]

#         if taxon.parent_taxon_id is None:
#             continue
#         parent = taxonomy.taxon_id_to_taxon[taxon.parent_taxon_id]
#         ascendents_class_ids = [parent.spatial_class_id]
#         while parent.parent_taxon_id is not None:
#             parent = taxonomy.taxon_id_to_taxon[parent.parent_taxon_id]
#             if parent.parent_taxon_id is not None:
#                 ascendents_class_ids.append(parent.spatial_class_id)
#         parent_taxon_id = taxon.parent_taxon_id
        
#         new_set = list(set(ascendents_class_ids + [cid]))
#         spatial_train_h3_dense.loc[row.name, "spatial_class_id"] = new_set



In [34]:
len(spatial_train_h3_dense.iloc[99999].spatial_class_id)

7

# make random samples

In [35]:
def make_samples(batch_size):
    rand_loc = np.random.uniform(size=(batch_size, 2))

    theta1 = 2.0 * math.pi * rand_loc[:, 0]
    theta2 = np.arccos(2.0 * rand_loc[:, 1] - 1.0)

    lat = 1.0 - 2.0 * theta2 / math.pi
    lng = (theta1 / math.pi) - 1.0

    return list(zip(lng, lat))



In [36]:
samples = make_samples(params["num_random_samples"])

scaled_samples = []
for sample in samples:
    lng, lat = sample
    lng = lng * 180
    lat = lat * 90
    scaled_samples.append((lng, lat))

negatives_df = pd.DataFrame(scaled_samples, columns=["lng", "lat"])
negatives_h3 = negatives_df.h3.geo_to_h3(params["h3_resolution"])
negatives_h3 = negatives_h3[~negatives_h3.index.duplicated(keep="first")]

# combine spatial data with empties

In [37]:
empty_valid_indices = list(
    set(negatives_h3.index).difference(
        set(spatial_train_h3_dense.index)
    )
)
negative_valid_h3 = negatives_h3[
    negatives_h3.index.isin(empty_valid_indices)
]

combined = pd.concat([negative_valid_h3, spatial_train_h3_dense])
combined["spatial_class_id"] = combined["spatial_class_id"].apply(
    lambda d: d if isinstance(d, list) else []
)
combined = combined[["spatial_class_id"]]
combined = combined.h3.h3_to_geo()
combined = combined.sort_index()
combined["lng"] = combined["geometry"].x
combined["lat"] = combined["geometry"].y
_ = combined.pop("geometry")

combined = combined


# layer in elevation

In [38]:
elevations = pd.read_csv(
    params["elevation_file"], index_col=combined.index.name
)

In [39]:
# normalize it
above_sea_level_norm = elevations.elevation[elevations.elevation > 0] / np.max(
    elevations.elevation[elevations.elevation > 0]
)
below_sea_level_norm = (
    elevations.elevation[elevations.elevation < 0]
    / np.min(elevations.elevation[elevations.elevation < 0])
) * -1
elev_normalized = pd.concat([below_sea_level_norm, above_sea_level_norm])
elev_norm_df = pd.DataFrame(
    {
        "elevation": elev_normalized,
    }
)

In [40]:
combined_with_elevation = pd.merge(
    combined,
    elev_norm_df,
    left_index=True,
    right_index=True,
)

In [41]:
assert combined_with_elevation.isna().any().any() == False

# write tfrecords

In [42]:
os.makedirs(
    os.path.join(params["export_dir"], "geo_spatial_grid_datasets"),
    exist_ok=True,
)

In [43]:
tfrecord_file = os.path.join(
    params["export_dir"],
    "geo_spatial_grid_datasets",
    "r{}_empty_cells_with_elevation_inner_nodes.tf".format(params["h3_resolution"]),
)
tfrecord_file

'/data-ssd/alex/datasets/vision-export-20240512050009-aka-2.14/geo_spatial_grid_datasets/r6_empty_cells_with_elevation_inner_nodes.tf'

In [44]:
if params["full_shuffle_before_tfrecords"]:
    print("shuffle all data before making tfrecords")
    combined_with_elevation = combined_with_elevation.sample(frac=1)

In [45]:
def create_tf_example(l0, l1, l2, l3, elevation, leaf_class_ids):
    tf_example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "l0": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l0])
                ),
                "l1": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l1])
                ),
                "l2": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l2])
                ),
                "l3": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l3])
                ),
                "elevation": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[elevation])
                ),
                "leaf_class_ids": tf.train.Feature(
                    int64_list=tf.train.Int64List(value=leaf_class_ids)
                ),
            }
        )
    )
    return tf_example



In [46]:
print("  writing tfrecords")
with tf.io.TFRecordWriter(tfrecord_file) as writer:
    total = len(combined_with_elevation)
    i = 0
    for _, row in combined_with_elevation.iterrows():
        if i % 100_000 == 0:
            print("  {} of {} written".format(i, total))
        i += 1

        lat = row["lat"]
        lng = row["lng"]

        norm_lat = lat / 90.0
        norm_lng = lng / 180.0
        norm_loc = np.array([norm_lng, norm_lat])
        encoded_loc = np.concatenate(
            [np.sin(norm_loc * math.pi), np.cos(norm_loc * math.pi)]
        )

        l0 = encoded_loc[0]
        l1 = encoded_loc[1]
        l2 = encoded_loc[2]
        l3 = encoded_loc[3]

        elevation = row["elevation"]
        cids = row["spatial_class_id"]
        example = create_tf_example(l0, l1, l2, l3, elevation, cids)
        writer.write(example.SerializeToString())
    writer.close()

  writing tfrecords
  0 of 942347 written
  100000 of 942347 written
  200000 of 942347 written
  300000 of 942347 written
  400000 of 942347 written
  500000 of 942347 written
  600000 of 942347 written
  700000 of 942347 written
  800000 of 942347 written
  900000 of 942347 written


In [47]:
!ls -lah {tfrecord_file}

-rw-rw-r-- 1 alex alex 663M Jul 29 06:27 /data-ssd/alex/datasets/vision-export-20240512050009-aka-2.14/geo_spatial_grid_datasets/r6_empty_cells_with_elevation_inner_nodes.tf
